#### Importing required libraries 

In [1]:
from utils import Hetero_Data_Processor_Filter_on_Test_since_first_post # Required class for GNN Incremental training
import pandas as pd
from torch import nn
from torch_geometric.nn import GATConv,to_hetero
import torch.nn.functional as F
import numpy as np
import mlflow
import torch
import torch.nn as nn
import warnings
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
warnings.filterwarnings("ignore")

#### Testing a single load 

In [2]:
event_name ="sydneysiege"

In [3]:
file_path_replies = f"replies_{event_name}.pkl"
file_path_posts = f"posts_{event_name}.pkl"


processor = Hetero_Data_Processor_Filter_on_Test_since_first_post(file_path_replies, file_path_posts, time_cut=15000)
data = processor.process()


In [4]:
data['id'].test_mask.sum()+data['id'].val_mask.sum()+data['id'].train_mask.sum()

tensor(1172)

In [5]:
data['id'].test_mask.sum()+data['id'].val_mask.sum()

tensor(352)

In [6]:
data['id'].test_mask.sum()

tensor(176)

In [7]:
data['id'].y = data['id'].y.long()

In [8]:
class GAT(torch.nn.Module):
    """
    Graph Attention Network (GAT) model with two GATConv layers and a final
    linear projection layer.

    This architecture applies graph attention mechanisms to learn contextual
    node embeddings based on graph connectivity. Dropout is applied after each
    GAT layer to help reduce overfitting.

    Parameters
    ----------
    dim_h : int
        Number of hidden attention heads/features in the first GAT layer.
    dim_i : int
        Number of intermediate output features of the second GAT layer.
    dim_out : int
        Output feature dimension, typically corresponding to the number of
        prediction classes or embedding size.

    Attributes
    ----------
    conv1 : GATConv
        First graph attention convolution layer with learned attention weights.
    conv2 : GATConv
        Second graph attention convolution layer that refines node embeddings.
    linear : nn.Linear
        Fully connected layer to project the learned embeddings to the output dimension.
    dropout : nn.Dropout
        Dropout layer applied after each convolution to reduce overfitting.

    Forward Inputs
    --------------
    x : torch.Tensor
        Node feature matrix of shape [num_nodes, num_features].
    edge_index : torch.LongTensor
        Graph edge index tensor of shape [2, num_edges] defining connectivity.

    Returns
    -------
    torch.Tensor
        Output node representations of shape [num_nodes, dim_out].
    """

    def __init__(self, dim_h,dim_i, dim_out):
        super().__init__()
        self.conv1 = GATConv((-1, -1), dim_h, add_self_loops=False)
        self.conv2 = GATConv(dim_h, dim_i, add_self_loops=False)
        self.linear = nn.Linear(dim_i, dim_out)
        self.dropout = nn.Dropout(p=0.4)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index).relu()
        h = self.dropout(h)
        h = self.conv2(h, edge_index).relu()
        h = self.dropout(h)
        h = self.linear(h)
        return h


In [9]:


def evaluate(model, data, mask_names):


    """
    Evaluate a graph classification model using masked subsets of the data.

    This function runs the model in evaluation mode, computes predictions,
    filters them using one or multiple masks from the input dataset, and
    returns common binary classification metrics.

    Parameters
    ----------
    model : torch.nn.Module
        Trained GNN model producing class logits from graph inputs.
    data : torch_geometric.data.HeteroData
        Heterogeneous graph data structure containing:
        - `x_dict`: dictionary of node feature matrices
        - `edge_index_dict`: dictionary of edge connectivity
        - `'id'` node type with attributes `y` and boolean masks
          (e.g., 'train_mask', 'val_mask', 'test_mask')
    mask_names : str or list[str]
        Name(s) of mask attributes to evaluate on. If multiple masks
        are provided, they are combined using logical OR.

    Returns
    -------
    tuple(float, float, float, float)
        A tuple containing:
        - acc : float
            Accuracy score.
        - precision : float
            Proportion of predicted positives that are correctly classified.
        - recall : float
            True positive rate.
        - auc : float
            ROC-AUC score based on predicted class probabilities.

    Notes
    -----
    - Metrics are computed only on masked nodes.
    - If ROC-AUC cannot be computed due to a single class present in labels,
      a value of 0.0 is returned.
    """

    model.eval()
    
    out = model(data.x_dict, data.edge_index_dict)['id']
    preds = out.argmax(dim=-1)
    labels = data['id'].y

    # --- Handle mask types ---
    if isinstance(mask, torch.Tensor):
        final_mask = mask

    elif isinstance(mask, str):
        final_mask = data['id'][mask]

    elif isinstance(mask, list):
        final_mask = torch.zeros_like(labels, dtype=torch.bool)
        for name in mask:
            final_mask |= data['id'][name]
    else:
        raise ValueError("mask must be a tensor, string, or list of strings")

    # --- Apply mask ---
    preds_masked = preds[final_mask]
    labels_masked = labels[final_mask]
    probs = out[final_mask][:, 1]  # class 1 probabilities

    # --- Metrics ---
    acc = accuracy_score(labels_masked.cpu(), preds_masked.cpu())
    precision = precision_score(labels_masked.cpu(), preds_masked.cpu(), zero_division=0)
    recall = recall_score(labels_masked.cpu(), preds_masked.cpu(), zero_division=0)

    try:
        auc = roc_auc_score(labels_masked.cpu(), probs.detach().cpu())
    except ValueError:
        auc = 0.0

    return acc, precision, recall, auc






#### Example  training

In [10]:

model = GAT(dim_h=64,dim_i=32, dim_out=2)
model = to_hetero(model, data.metadata(), aggr='sum')

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data, model = data.to(device), model.to(device)

In [11]:
def evaluate_metrics(model, data, mask):


    """
    Compute evaluation metrics for a model on a masked node subset.

    The function performs prediction using the trained model, extracts
    predictions over a specific boolean mask, and computes standard
    classification metrics including accuracy, macro precision, macro
    recall, and ROC-AUC.

    Parameters
    ----------
    model : torch.nn.Module
        Trained GNN model that outputs logits for the `'id'` node type.
    data : torch_geometric.data.HeteroData
        Heterogeneous graph data containing node features, edge connections,
        labels, and a boolean mask to filter evaluation nodes.
    mask : torch.Tensor or list[bool]
        Boolean mask selecting the subset of nodes to evaluate.

    Returns
    -------
    tuple(float, float, float, float)
        A tuple of:
        - acc : float
        
            Accuracy score.
        - prec : float
            Macro-averaged precision.
        - recall : float
            Macro-averaged recall.
        - auc : float
            ROC-AUC score based on probability of the positive class.

    Notes
    -----
    - Evaluation is performed inside a `torch.no_grad()` block to disable gradient tracking.
    - If ROC-AUC computation fails (e.g., only one class present), a value of 0.0 is returned.
    """


    model.eval()
    with torch.no_grad():
        out = model(data.x_dict, data.edge_index_dict)['id']
        preds = out.argmax(dim=1)
        probs = out[:, 1]  # Probability of class 1

    true = data['id'].y[mask]
    pred = preds[mask]
    prob = probs[mask]

    acc = accuracy_score(true.cpu(), pred.cpu())
    prec = precision_score(true.cpu(), pred.cpu(), average='macro', zero_division=0)
    recall = recall_score(true.cpu(), pred.cpu(), average='macro', zero_division=0)
    try:
        auc = roc_auc_score(true.cpu(), prob.cpu())
    except:
        auc = 0.0

    # --- Build DataFrame ---
    df = pd.DataFrame({
        "node_id": mask.nonzero(as_tuple=False).view(-1).cpu().numpy(),
        "true_label": true.cpu().numpy(),
        "prob": prob.cpu().numpy()
    })

    return acc, prec, recall, auc, df

In [12]:
for epoch in range(1, 101):
            model.train()
            optimizer.zero_grad()
            out = model(data.x_dict, data.edge_index_dict)['id']
            mask = data['id'].train_mask
            loss = F.cross_entropy(out[mask], data['id'].y[mask])
            loss.backward()
            optimizer.step()

            if epoch % 100 == 0:
                 train_acc, train_prec, train_recall, train_auc= evaluate(model, data, data['id'].train_mask)
                 print(f"[Epoch {epoch}] Train Loss: {loss:.4f} | Train Recall: {train_recall:.4f} | Train Auc: {train_auc:.4f}")

# Final evaluation on val + test
final_mask = data['id'].val_mask | data['id'].test_mask
acc, prec, recall, auc,df_probs = evaluate_metrics(model, data, final_mask)
print(f"[Final Val+Test] Acc: {acc:.4f} | Prec: {prec:.4f} | Recall: {recall:.4f} | AUC: {auc:.4f}")

[Epoch 100] Train Loss: 0.5062 | Train Recall: 0.8682 | Train Auc: 0.9073
[Final Val+Test] Acc: 0.7244 | Prec: 0.7276 | Recall: 0.7622 | AUC: 0.7805


In [13]:
def compute_metrics_custom(df, prob_col='prob', target_col='rumour'):
    df = df.copy()
    df = df.sort_values(prob_col, ascending=False).reset_index(drop=True)

    total_frauds = df[target_col].sum()
    total_records = df.shape[0]
    n = len(df)

    # ✅ Bins from 1% to 100% in 1% steps
    bins = np.arange(0.01, 1.01, 0.01)

    results = []

    for p in bins:
        cutoff = int(np.ceil(n * p))
        subset = df.iloc[:cutoff]

        frauds = subset[target_col].sum()
        records = len(subset)

        results.append({
            'percentile': round(p * 100, 0),
            'records': records,
            'frauds_captured': frauds,
            'capture_rate': frauds / total_frauds if total_frauds > 0 else 0,
            'bad_rate': frauds / records if records > 0 else 0,
            'false_positive_rate': (records - frauds) / total_records if records > 0 else 0
        })

    df_out = (
        pd.DataFrame(results)
        .drop_duplicates(subset='percentile')
        .sort_values('percentile')
        .reset_index(drop=True)
    )

    return df_out

**Creating evaluate_metrics function to assess classification when new posts are created**

#### Setting MLflow Experiment

In [10]:
from datetime import date

today = date.today()
formatted_today = today.strftime("%Y-%m-%d")

In [11]:

mlflow.set_experiment(f"GAT {formatted_today} {event_name}")

2026/07/05 18:06:49 INFO mlflow.tracking.fluent: Experiment with name 'GAT 2026-07-05 ferguson' does not exist. Creating a new experiment.


<Experiment: artifact_location='/workspaces/rumour-detection-gnn/mlruns/125', creation_time=1783274809024, experiment_id='125', last_update_time=1783274809024, lifecycle_stage='active', name='GAT 2026-07-05 ferguson', tags={}, workspace='default'>

#### Loading dataset statistics to get the final time cut 

In [7]:
import pandas as pd
df_posts_by_time_cut = pd.read_pickle(f'replies_{event_name}.pkl')

df_posts_by_time_cut['min_since_fst_post'] = round(
            (df_posts_by_time_cut['time'] - df_posts_by_time_cut['time'].min()).dt.total_seconds() / 60, 2)


In [8]:
df_metrics = df_posts_by_time_cut[['id','time','rumour','min_since_fst_post']].drop_duplicates().sort_values(by='time')

In [9]:
file_path_replies = f"replies_{event_name}.pkl"
file_path_posts = f"posts_{event_name}.pkl"


processor = Hetero_Data_Processor_Filter_on_Test_since_first_post(file_path_replies, file_path_posts, time_cut=1000)
data = processor.process()


In [10]:

start = df_metrics.iloc[int(data['id'].train_mask.sum()):].min_since_fst_post.min()
end = df_metrics.iloc[int(data['id'].train_mask.sum()):].min_since_fst_post.max()
duration = end-start
experiment_time= duration+60



In [21]:
start

np.float64(2542.68)

In [22]:
end

np.float64(4530.47)

In [23]:
experiment_time

np.float64(2047.7900000000004)

#### Experiment all posts

In [28]:
previous_node_count = 0  # Start with no nodes

for time_cut in np.linspace(15, int(experiment_time), 50):
    time_cut = int(time_cut)
    print(f"\n=== Time Cut: {time_cut} minutes ===")

    processor = Hetero_Data_Processor_Filter_on_Test_since_first_post(file_path_replies, file_path_posts, time_cut=time_cut)
    data = processor.process()
    data['id'].y = data['id'].y.long()

    current_node_count = data['id'].x.shape[0]
    new_node_indices = np.arange(previous_node_count, current_node_count)
    previous_node_count = current_node_count

    # Set up model and training

    model = GAT(dim_h=64,dim_i=32, dim_out=2)
    model = to_hetero(model, data.metadata(), aggr='sum')
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data, model = data.to(device), model.to(device)
    
    with mlflow.start_run(run_name=f"time_cut_{time_cut}"):
        for epoch in range(1, 201):
            model.train()
            optimizer.zero_grad()
            out = model(data.x_dict, data.edge_index_dict)['id']
            mask = data['id'].train_mask
            loss = F.cross_entropy(out[mask], data['id'].y[mask])
            loss.backward()
            optimizer.step()

            if epoch % 100 == 0:
                 train_acc, train_prec, train_recall, train_auc= evaluate(model, data, data['id'].train_mask)
                 print(f"[Epoch {epoch}] Train Loss: {loss:.4f} | Train Recall: {train_recall:.4f} | Train Precision: {train_prec:.4f}")
    
        # Evaluate all predictions
        model.eval()
        with torch.no_grad():
            out = model(data.x_dict, data.edge_index_dict)['id']
            preds = out.argmax(dim=1)
            probs = out[:, 1]
    
        # New instances in val/test set
        val_test_mask = (data['id'].val_mask | data['id'].test_mask).cpu().numpy()
        new_instance_mask = np.zeros_like(val_test_mask, dtype=bool)
        new_instance_mask[new_node_indices] = True
        final_mask = new_instance_mask & val_test_mask
    
        if final_mask.sum() > 0:
            # Compute metrics
            true_new = data['id'].y.cpu().numpy()[final_mask]
            pred_new = preds.cpu().numpy()[final_mask]
            prob_new = probs.cpu().numpy()[final_mask]
        
            new_precision = precision_score(true_new, pred_new, average='macro', zero_division=0)
            new_recall = recall_score(true_new, pred_new, average='macro', zero_division=0)
            new_acc = accuracy_score(true_new, pred_new)
        else:
            new_precision = 0
            new_recall = 0
            new_acc =0
            print("No new instances to evaluate.")
            
    
        # Compute metrics
        
        all_eval_mask = data['id'].val_mask | data['id'].test_mask
        acc, prec, recall, auc,df_probs = evaluate_metrics(model, data, all_eval_mask)
       # _, _, _, _,df_probs_new = evaluate_metrics(model, data, torch.tensor(final_mask))
        current_df_metrics = df_metrics.iloc[int(data['id'].train_mask.sum()):int(data['id'].train_mask.sum()\
                                                                          +data['id'].val_mask.sum()+data['id'].test_mask.sum())]
        current_df_metrics['prob'] = df_probs['prob'].to_numpy()
        df_metrics_by_bucket = compute_metrics_custom(current_df_metrics)


        #new_df_metrics = current_df_metrics.iloc[-final_mask.sum():]
        #new_df_metrics['prob'] = df_probs_new.to_numpy()
        #new_df_metrics_by_bucket = compute_metrics_custom(new_df_metrics)
        
        print(f"[Final Val+Test] Acc: {acc:.4f} | Prec: {prec:.4f} | Recall: {recall:.4f} | AUC: {auc:.4f}")
    
    
        print(f"New Instances: {final_mask.sum()}")
        print(f"New Precision: {new_precision:.4f} | New Recall: {new_recall:.4f}")

        mlflow.log_metric("new_posts", final_mask.sum())
        
        mlflow.log_metric("final_precision", prec)
        mlflow.log_metric("final_recall", recall)
        mlflow.log_metric("final_auc", auc)
        mlflow.log_metric("final_acc", acc)
        mlflow.log_param('new_post',False)
        df_metrics_by_bucket.to_csv(f"metrics_by_bucket_{event_name}_GAT.csv", index=False)
        mlflow.log_artifact(f"metrics_by_bucket_{event_name}_GAT.csv")

        mlflow.log_metric("curr_precision", new_precision)
        mlflow.log_metric("curr_recall", new_recall)
        mlflow.log_metric("curr_acc", new_acc)

        mlflow.log_metric("time_cut", time_cut)





=== Time Cut: 15 minutes ===
[Epoch 100] Train Loss: 0.4423 | Train Recall: 0.9057 | Train Precision: 0.7869
[Epoch 200] Train Loss: 0.3323 | Train Recall: 0.9245 | Train Precision: 1.0000
[Final Val+Test] Acc: 0.5000 | Prec: 0.3333 | Recall: 0.3333 | AUC: 0.6667
New Instances: 4
New Precision: 0.3333 | New Recall: 0.3333

=== Time Cut: 56 minutes ===
[Epoch 100] Train Loss: 0.4020 | Train Recall: 0.8679 | Train Precision: 0.9200
[Epoch 200] Train Loss: 0.1360 | Train Recall: 0.9434 | Train Precision: 1.0000
[Final Val+Test] Acc: 0.7692 | Prec: 0.7639 | Recall: 0.7375 | AUC: 0.9250
New Instances: 9
New Precision: 0.9167 | New Recall: 0.8750

=== Time Cut: 97 minutes ===
[Epoch 100] Train Loss: 0.3892 | Train Recall: 0.8868 | Train Precision: 0.8545
[Epoch 200] Train Loss: 0.2065 | Train Recall: 0.9245 | Train Precision: 1.0000
[Final Val+Test] Acc: 0.7059 | Prec: 0.6591 | Recall: 0.6750 | AUC: 0.8000
New Instances: 4
New Precision: 0.5000 | New Recall: 0.2500

=== Time Cut: 139 minute

In [12]:
new_posts_times = np.sort(
    np.unique(
        np.ceil(
            df_metrics[
                (df_metrics.min_since_fst_post >= start) &
                (df_metrics.min_since_fst_post <= end)
            ].min_since_fst_post - start
        )
    )
)

In [13]:
new_posts_times = [col for col in new_posts_times if col > 10]

In [ ]:

previous_node_count = 0  # Start with no nodes

for time_cut in new_posts_times:
    print(f"\n=== Time Cut: {time_cut} minutes ===")

    processor = Hetero_Data_Processor_Filter_on_Test_since_first_post(file_path_replies, file_path_posts, time_cut=time_cut)
    data = processor.process()
    data['id'].y = data['id'].y.long()

    current_node_count = data['id'].x.shape[0]
    new_node_indices = np.arange(previous_node_count, current_node_count)
    previous_node_count = current_node_count

    # Set up model and training

    model = GAT(dim_h=64,dim_i=32, dim_out=2)
    model = to_hetero(model, data.metadata(), aggr='sum')
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data, model = data.to(device), model.to(device)
    
    with mlflow.start_run(run_name=f"time_cut_{time_cut}"):
        for epoch in range(1, 201):
            model.train()
            optimizer.zero_grad()
            out = model(data.x_dict, data.edge_index_dict)['id']
            mask = data['id'].train_mask
            loss = F.cross_entropy(out[mask], data['id'].y[mask])
            loss.backward()
            optimizer.step()

            if epoch % 100 == 0:
                 train_acc, train_prec, train_recall, train_auc= evaluate(model, data, data['id'].train_mask)
                 print(f"[Epoch {epoch}] Train Loss: {loss:.4f} | Train Recall: {train_recall:.4f} | Train Precision: {train_prec:.4f}")
    
        # Evaluate all predictions
        model.eval()
        with torch.no_grad():
            out = model(data.x_dict, data.edge_index_dict)['id']
            preds = out.argmax(dim=1)
            probs = out[:, 1]
    
        # New instances in val/test set
        val_test_mask = (data['id'].val_mask | data['id'].test_mask).cpu().numpy()
        new_instance_mask = np.zeros_like(val_test_mask, dtype=bool)
        new_instance_mask[new_node_indices] = True
        final_mask = new_instance_mask & val_test_mask
    
        if final_mask.sum() > 0:
            # Compute metrics
            true_new = data['id'].y.cpu().numpy()[final_mask]
            pred_new = preds.cpu().numpy()[final_mask]
            prob_new = probs.cpu().numpy()[final_mask]
        
            new_precision = precision_score(true_new, pred_new, average='macro', zero_division=0)
            new_recall = recall_score(true_new, pred_new, average='macro', zero_division=0)
            new_acc = accuracy_score(true_new, pred_new)
        else:
            new_precision = 0
            new_recall = 0
            new_acc =0
            print("No new instances to evaluate.")
            continue
            
    
        # Compute metrics
        
        all_eval_mask = data['id'].val_mask | data['id'].test_mask
        acc, prec, recall, auc,df_probs = evaluate_metrics(model, data, all_eval_mask)
        _, _, _, _,df_probs_new = evaluate_metrics(model, data, torch.tensor(final_mask))
        current_df_metrics = df_metrics.iloc[int(data['id'].train_mask.sum()):int(data['id'].train_mask.sum()\
                                                                          +data['id'].val_mask.sum()+data['id'].test_mask.sum())]
        current_df_metrics['prob'] = df_probs['prob'].to_numpy()
        df_metrics_by_bucket = compute_metrics_custom(current_df_metrics)


        new_df_metrics = current_df_metrics.iloc[-final_mask.sum():]
        new_df_metrics['prob'] = df_probs_new.to_numpy()
        new_df_metrics_by_bucket = compute_metrics_custom(new_df_metrics)
        
        print(f"[Final Val+Test] Acc: {acc:.4f} | Prec: {prec:.4f} | Recall: {recall:.4f} | AUC: {auc:.4f}")
    
    
        print(f"New Instances: {final_mask.sum()}")
        print(f"New Precision: {new_precision:.4f} | New Recall: {new_recall:.4f}")

        mlflow.log_metric("new_posts", final_mask.sum())
        
        mlflow.log_metric("final_precision", prec)
        mlflow.log_metric("final_recall", recall)
        mlflow.log_metric("final_auc", auc)
        mlflow.log_metric("final_acc", acc)
        mlflow.log_param('new_post',True)
        new_df_metrics_by_bucket.to_csv(f"metrics_by_bucket_new_posts_{event_name}_GAT.csv", index=False)
        mlflow.log_artifact(f"metrics_by_bucket_new_posts_{event_name}_GAT.csv")

        mlflow.log_metric("curr_precision", new_precision)
        mlflow.log_metric("curr_recall", new_recall)
        mlflow.log_metric("curr_acc", new_acc)

        mlflow.log_metric("time_cut", time_cut)





=== Time Cut: 11.0 minutes ===
[Epoch 100] Train Loss: 0.3958 | Train Recall: 0.0351 | Train Precision: 1.0000
